In [ ]:
from typing import Annotated, Sequence
from pydantic import BaseModel, Field
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver  # ←←

# === 1. 定义状态 Schema (Pydantic) ===
class AgentState(BaseModel):
    messages: Annotated[Sequence[str], operator.add] = Field(default_factory=list)
    step_count: int = 0


# === 2. 节点函数 ===
def node1(state: AgentState) -> dict:
    print(f"[Node1] 当前 step_count: {state.step_count}")
    new_message = f"Hello from node1 at step {state.step_count + 1}"
    return {
        "messages": [new_message],
        "step_count": state.step_count + 1,
    }


def node2(state: AgentState) -> dict:
    print(f"[Node2] 当前 step_count: {state.step_count}")
    new_message = f"Goodbye from node2 at step {state.step_count + 1}"
    return {
        "messages": [new_message],
        "step_count": state.step_count + 1,
    }


# === 3. 构建图 ===
builder = StateGraph(AgentState)

builder.add_node("node1", node1)
builder.add_node("node2", node2)

builder.add_edge(START, "node1")
builder.add_edge("node1", "node2")
builder.add_edge("node2", END)

# 编译时必须传入 checkpointer
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)  # ←← 启用 checkpointer

# === 4. 执行图并传入 config（包含 thread_id）===
config = {"configurable": {"thread_id": "123"}}
result = graph.invoke(
    {"messages": ["Initial input"], "step_count": 0},
    config=config
)

print("\n 最终输出:")
print(result)

# === 5. 获取状态历史（快照）===
print("\n 状态变更历史（快照）:")
state_history = list(graph.get_state_history(config))
for state in state_history:
    print(state)

可以根据检查点 ID 回到任意一个状态，并从那里重新启动 Agent

In [ ]:
config = {"configurable":{'thread_id': '123',  'checkpoint_id': '1f090bb7-f882-6a9b-8001-220d0d477fdb'}}

checkpoint_snapshot = graph.get_state(config)

graph.invoke({"messages": ["Initial input"], "step_count": 0}, config=config)

In [ ]:
!pip install -qU redis aioredis

In [ ]:
import re
from typing import Annotated, Sequence, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver


# 1. 定义状态
class GraphState(TypedDict):
    messages: Annotated[Sequence, add_messages]
    retry_count: int


# 2. 输入校验函数
def is_valid_order_id(text: str) -> bool:
    return bool(re.fullmatch(r"\d{10,12}", text))


def validate_input(state: GraphState) -> Literal["valid", "invalid"]:
    last_message = state["messages"][-1]
    user_input = last_message.content.strip()
    if is_valid_order_id(user_input):
        return "valid"
    return "invalid"


# 3. 节点函数
def receive_input(state: GraphState):
    user_input = state["messages"][-1].content.strip()
    return {"order_id": user_input}


def query_order(state: GraphState):
    order_id = state["messages"][-1].content.strip()
    print(f"查询订单: {order_id}")
    return {
        "messages": ["订单状态: 已发货"],
        "status": "success"
    }


def handle_invalid(state: GraphState):
    retry = state.get("retry_count", 0)
    if retry >= 2:
        reply = "输入错误次数过多，会话结束。"
        return {"messages": [reply], "retry_count": retry + 1}

    reply = "订单号不合法，请输入10到12位数字。"
    return {"messages": [reply], "retry_count": retry + 1}


# 4. 构建图
builder = StateGraph(GraphState)

builder.add_node("receive_input", receive_input)
builder.add_node("query_order", query_order)
builder.add_node("handle_invalid", handle_invalid)

# 条件跳转
builder.add_conditional_edges(
    START,
    validate_input,
    {
        "valid": "query_order",
        "invalid": "handle_invalid"
    }
)

# 如果无效，最多重试2次
builder.add_conditional_edges(
    "handle_invalid",
    lambda s: "receive_input" if s["retry_count"] < 2 else END
)

builder.add_edge("receive_input", "query_order")
builder.add_edge("query_order", END)

# 编译图，使用 MemorySaver 记录状态历史
memory_saver = MemorySaver()
app = builder.compile(checkpointer=memory_saver)


# 5. 测试运行
config = {"configurable": {"thread_id": "123"}}

# 第一次：非法输入
app.invoke(
    {"messages": ["abc123"]},
    config=config
)

# 查看状态历史
history = app.get_state_history(config)
for snapshot in history:
    print("Messages:", snapshot.values["messages"])
    print("Retry count:", snapshot.values.get("retry_count", 0))
    print("---")

查询订单: 订单号不合法，请输入10到12位数字。
Messages: [HumanMessage(content='abc123', additional_kwargs={}, response_metadata={}, id='f308c0ac-808b-4a0e-867b-7054cfe6067a'), HumanMessage(content='订单号不合法，请输入10到12位数字。', additional_kwargs={}, response_metadata={}, id='c7ec3a08-3bb6-437a-b976-91e190ede239'), HumanMessage(content='订单状态: 已发货', additional_kwargs={}, response_metadata={}, id='d6d26295-cc11-43dd-8033-15cd6da5470c')]
Retry count: 1
---
Messages: [HumanMessage(content='abc123', additional_kwargs={}, response_metadata={}, id='f308c0ac-808b-4a0e-867b-7054cfe6067a'), HumanMessage(content='订单号不合法，请输入10到12位数字。', additional_kwargs={}, response_metadata={}, id='c7ec3a08-3bb6-437a-b976-91e190ede239')]
Retry count: 1
---
Messages: [HumanMessage(content='abc123', additional_kwargs={}, response_metadata={}, id='f308c0ac-808b-4a0e-867b-7054cfe6067a'), HumanMessage(content='订单号不合法，请输入10到12位数字。', additional_kwargs={}, response_metadata={}, id='c7ec3a08-3bb6-437a-b976-91e190ede239')]
Retry count: 1
---
Messages:

In [20]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END, add_messages
from langchain_community.chat_models import ChatTongyi
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver

In [21]:
import os
class GraphState(TypedDict):
    messages: Annotated[Sequence, add_messages]
    current_model: str

# 主模型（故意设错 API Key 来模拟调用失败）
primary_model = ChatTongyi(
    model_name="qwen-plus",
    api_key="invalid_key_for_test",  # 强制报错
    temperature=0.7,
)

# 备用模型（使用 DASHSCOPE_API_KEY 环境变量）
backup_model = ChatTongyi(
    model_name="qwen-max",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    temperature=0.7,
)

In [22]:
def call_llm(state):
    messages = state["messages"]
    used_model = ""
    response_message = None

    # 尝试主模型
    try:
        print("尝试调用主模型 (qwen-plus)...")
        response = primary_model.invoke(messages)
        response_message = response
        used_model = "qwen-plus"
    except Exception as e:
        print(f"主模型失败: {type(e).__name__}: {str(e)}")
        print("切换到备用模型 (qwen-max)")

        # 尝试备用模型
        try:
            response = backup_model.invoke(messages)
            response_message = response
            used_model = "qwen-max"
        except Exception as e2:
            print(f"备用模型也失败: {str(e2)}")
            return {
                "messages": [AIMessage(content="服务暂时不可用，请稍后再试。")],
                "current_model": "none"
            }

    return {
        "messages": [AIMessage(content=response_message.content)],
        "current_model": used_model
    }

builder = StateGraph(GraphState)
builder.add_node("call_llm", call_llm)
builder.add_edge(START, "call_llm")
builder.add_edge("call_llm", END)

# 启用检查点，用于后续查看状态历史
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

In [24]:
# 测试

config = {"configurable": {"thread_id": "test_001"}}

inputs = {
    "messages": [HumanMessage(content="解释一下什么是机器学习")],
    "current_model": ""
}

result = app.invoke(inputs, config=config)

尝试调用主模型 (qwen-plus)...
主模型失败: ValueError: request_id: 1748f5c0-285a-445e-9e10-11c396213408 
 status_code: 401 
 code: InvalidApiKey 
 message: Invalid API-key provided.
切换到备用模型 (qwen-max)


In [25]:
print("最终使用的模型:", result["current_model"])

for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print("AI 回复:")
        print(msg.content)

最终使用的模型: qwen-max
AI 回复:
机器学习是一种人工智能技术，它使计算机能够在没有明确编程的情况下从数据中学习并改进其性能。简而言之，就是让计算机通过学习数据中的模式和规律来自动地完成特定任务，并且随着经验的积累而不断优化自己的表现。

在机器学习过程中，通常会使用算法来分析输入的数据（训练集），从中识别出有用的模式或特征，然后基于这些信息构建模型。这个模型可以用来预测新数据的结果或者做出决策。根据不同的应用场景，机器学习主要分为以下几类：

1. **监督学习**：在这种类型的学习中，算法通过已知正确答案的数据集进行训练，目的是学会如何将输入映射到正确的输出。常见的应用包括分类（如垃圾邮件检测）和回归（如房价预测）问题。
   
2. **无监督学习**：与监督学习不同，在这种情况下，训练数据没有标签信息。算法需要自己发现数据中的结构或分布情况。聚类（比如客户细分）、降维等是无监督学习的一些例子。
   
3. **半监督学习**：结合了有标签和无标签的数据来进行学习。这种方法适用于获取大量未标记数据相对容易但获得准确标签成本较高的场景。
   
4. **强化学习**：这是一种通过试错的方式学习的方法。代理（agent）在一个环境中采取行动以最大化某种形式的累积奖励。强化学习广泛应用于游戏、机器人控制等领域。

机器学习的应用非常广泛，涵盖了自然语言处理、图像识别、推荐系统等多个领域，并且正逐渐成为许多行业提高效率、创新服务的关键技术之一。
AI 回复:
机器学习是一种人工智能的分支，它使计算机系统能够通过数据和经验自动改进其性能，而不需要显式编程。简单来说，就是让计算机从数据中学习模式，并利用这些模式来做出预测或决策。

以下是机器学习的一些关键概念和类型：

### 关键概念
1. **数据**：机器学习的基础是数据。数据可以是结构化的（如数据库中的表格）或非结构化的（如文本、图像、音频等）。
2. **模型**：模型是从数据中学习到的规则或模式的表示。它可以是一个数学公式、一个算法或一组参数。
3. **训练**：训练是指使用数据集来调整模型的参数，使其能够更好地拟合数据。
4. **预测**：训练完成后，模型可以用来对新的、未见过的数据进行预测或分类。
5. **评估**：通过一些指标（如准确率、召回率、F1分数等）来衡量模型在新数据上的